In [15]:
import tiktoken
text = "tiktoken is great!"
encoding = tiktoken.get_encoding("gpt2")
tokenized_text = encoding.encode(text, allowed_special="all")
print(tokenized_text)
decoded_text = encoding.decode(tokenized_text)
print(decoded_text)

[83, 1134, 30001, 318, 1049, 0]
tiktoken is great!


In [16]:
text = "tiktoken is great! <|endoftext|>"
tokenized_text = encoding.encode(text, allowed_special="all")
print(tokenized_text)
decoded_text = encoding.decode(tokenized_text)
print(decoded_text)

[83, 1134, 30001, 318, 1049, 0, 220, 50256]
tiktoken is great! <|endoftext|>


In [17]:
encoding.n_vocab

50257

In [18]:
with open("the-verdict.txt") as f:
    text = f.read()

print(len(text))


20479


In [19]:
import re
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result[:100])
text = " ".join(result)
print(text[:100])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter', '--']
I HAD always thought Jack Gisburn rather a cheap genius -- though a good fellow enough -- so it was 


In [20]:
import torch
from torch.utils.data import Dataset, DataLoader

In [21]:
class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
        # input = [0,1,2,3]
        # target = [1,2,3,4]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [22]:
def create_dataloader(text, batch_size=8, max_length=4, 
                        stride=2 , shuffle=True, drop_last=True,
                        num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDataset(text, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [23]:
dataloader = create_dataloader(text)
print(len(dataloader))

319


In [24]:
for input_ids, target_ids in dataloader:
    print(input_ids)
    print(target_ids)
    break

tensor([[  284,   467,   319, 12036],
        [  256,  1775,   257,  2060],
        [ 1139,  2063,   262,   640],
        [ 1204,   764,   366,  2094],
        [  764,   314,  2391,   531],
        [  582,   837,   290,   673],
        [ 6000,  1517,   373,   284],
        [10414,   738,   285, 23968]])
tensor([[  467,   319, 12036,  2162],
        [ 1775,   257,  2060,   530],
        [ 2063,   262,   640,   407],
        [  764,   366,  2094,   705],
        [  314,  2391,   531,   314],
        [  837,   290,   673,  1297],
        [ 1517,   373,   284,   766],
        [  738,   285, 23968,  4891]])


In [25]:
def embedding( dataloader, vocab_size, output_dim):
    embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
    all_embedded_inputs = []

    for input_ids, _ in dataloader:
        embedded_input = embedding_layer(input_ids)
        all_embedded_inputs.append(embedded_input)

    return torch.cat(all_embedded_inputs, dim=0)

In [26]:
data = embedding(dataloader, 50257, 256)
print(data.shape)

torch.Size([2552, 4, 256])


In [27]:
print(data[0])

tensor([[-0.7119,  0.5683, -0.3481,  ..., -1.9840,  1.0198, -0.8865],
        [ 0.1094, -0.0201,  0.9634,  ...,  0.0044, -0.2818,  0.4112],
        [ 0.6636,  0.6962, -0.6412,  ...,  0.2647,  1.2108, -0.5755],
        [ 0.1678,  0.0894, -0.6766,  ...,  0.2682,  0.0753, -1.2021]],
       grad_fn=<SelectBackward0>)


In [28]:
context_length = 4
output_dim = 256
max_length = context_length

In [29]:
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

print(pos_embeddings)

torch.Size([4, 256])
tensor([[-1.2962,  0.2146,  1.0033,  ..., -0.9124, -0.5351,  1.4886],
        [-0.4524,  0.2077, -1.5832,  ..., -0.6721,  0.3441, -1.1896],
        [-1.5803, -0.5803,  0.4645,  ...,  1.3388, -0.8162, -0.1508],
        [-0.5575,  0.4293,  0.7604,  ...,  2.1511, -1.6887,  0.9674]],
       grad_fn=<EmbeddingBackward0>)


In [30]:
embedded_inputs = data + pos_embeddings
print(embedded_inputs.shape)

torch.Size([2552, 4, 256])


In [38]:
print(embedded_inputs[6])

print(data[6])

tensor([[ 0.2267,  0.5504,  1.1760,  ..., -0.9817, -1.9680,  0.8005],
        [-1.2129, -1.8439, -2.3818,  ...,  0.0254, -0.3927, -3.1189],
        [ 0.6144,  1.1028, -0.2264,  ...,  1.9857, -0.5288, -0.0532],
        [-0.0616,  0.7071, -0.1822,  ...,  1.4886, -1.1216,  1.4077]],
       grad_fn=<SelectBackward0>)
tensor([[ 1.5229,  0.3358,  0.1727,  ..., -0.0693, -1.4330, -0.6881],
        [-0.7605, -2.0517, -0.7986,  ...,  0.6975, -0.7368, -1.9293],
        [ 2.1947,  1.6830, -0.6908,  ...,  0.6469,  0.2874,  0.0977],
        [ 0.4959,  0.2779, -0.9427,  ..., -0.6625,  0.5671,  0.4403]],
       grad_fn=<SelectBackward0>)


In [39]:
print(pos_embeddings+data[6])

tensor([[ 0.2267,  0.5504,  1.1760,  ..., -0.9817, -1.9680,  0.8005],
        [-1.2129, -1.8439, -2.3818,  ...,  0.0254, -0.3927, -3.1189],
        [ 0.6144,  1.1028, -0.2264,  ...,  1.9857, -0.5288, -0.0532],
        [-0.0616,  0.7071, -0.1822,  ...,  1.4886, -1.1216,  1.4077]],
       grad_fn=<AddBackward0>)
